# Clase 232 — Portafolio público: generador end-to-end

Esta clase es más de **producto y comunicación** que de código ML. El notebook es un **utilitario práctico**: define el schema de un proyecto, genera los `.md` del portafolio, valida que cumpla mínimos, y emite los stubs de `mkdocs.yml` + workflow de Pages.

Solo stdlib + `jinja2` opcional. Seed 42. Self-contained.

In [ ]:
from dataclasses import dataclass, field, asdict
from typing import Optional
import textwrap, random

random.seed(42)

try:
    from jinja2 import Template
    HAS_JINJA = True
except ImportError:
    HAS_JINJA = False
print(f'jinja2 disponible: {HAS_JINJA} (no es bloqueante, usamos f-strings de fallback)')

@dataclass
class Project:
    title: str
    slug: str
    problem: str            # 1-2 frases en lenguaje humano
    approach: str           # 2-3 frases técnicas
    metrics: dict           # {'metric_name': value}
    stack: list             # ['python', 'pytorch', ...]
    repo_url: str
    demo_url: Optional[str] = None
    blog_url: Optional[str] = None
    limitations: list = field(default_factory=list)
    hero_image: Optional[str] = None

## 1. Los 3 capstones como datos

Cada proyecto es un `Project` con campos llenos. Esto es lo que el portafolio renderiza.

In [ ]:
projects = [
    Project(
        title='Clasificador de sentimiento sobre reseñas en español',
        slug='nlp-sentimiento-es',
        problem='Detectar polaridad (pos/neg/neu) en reseñas de e-commerce en español rioplatense, donde modelos multilingües genéricos fallan con el slang local.',
        approach='Fine-tuning de `bertin-base` con LoRA sobre 12k reseñas etiquetadas. Servido con FastAPI + ONNX runtime; latencia p95 < 80ms.',
        metrics={'macro_f1': 0.847, 'accuracy': 0.881, 'latency_p95_ms': 78},
        stack=['python', 'pytorch', 'transformers', 'peft', 'fastapi', 'onnx', 'docker'],
        repo_url='https://github.com/usuario/capstone-nlp-es',
        demo_url='https://huggingface.co/spaces/usuario/sentimiento-es',
        blog_url='https://usuario.github.io/blog/2026/sentimiento-es/',
        limitations=['Sesgo regional: dataset 70% Argentina/Uruguay.', 'No detecta sarcasmo (F1 = 0.42 en subset sarcasm).', 'Solo texto, no maneja emojis.'],
        hero_image='assets/nlp-hero.gif',
    ),
    Project(
        title='Detector de defectos visuales en línea de producción',
        slug='cv-defect-detection',
        problem='Identificar 5 tipos de defecto (rayón, abolladura, mancha, fractura, contaminación) en piezas metálicas a 30 FPS sobre cámara industrial.',
        approach='Transfer learning de YOLOv8n sobre 4.2k imágenes etiquetadas (CVAT). Augmentación con Albumentations. Exportado a TensorRT para edge inference.',
        metrics={'mAP_50': 0.912, 'mAP_50_95': 0.687, 'fps_jetson_nano': 34},
        stack=['python', 'pytorch', 'ultralytics', 'opencv', 'albumentations', 'tensorrt'],
        repo_url='https://github.com/usuario/capstone-cv-defects',
        demo_url='https://usuario-cv-defects.streamlit.app',
        blog_url='https://usuario.github.io/blog/2026/cv-defects/',
        limitations=['Dataset de un solo proveedor — sin garantía de generalizar a otra fábrica.', 'No funciona con iluminación tipo flash.', 'Falsos positivos en piezas con grafismo de fábrica.'],
        hero_image='assets/cv-hero.png',
    ),
    Project(
        title='Predicción de churn en SaaS B2B',
        slug='tabular-churn',
        problem='Anticipar con 30 días de antelación qué cuentas B2B cancelarán suscripción, para priorizar account managers.',
        approach='LightGBM sobre 80 features de uso + facturación (ventana 90 días). Calibración isotónica. Explainability con SHAP. Métrica de negocio: recall@top-decile.',
        metrics={'auc_roc': 0.873, 'recall_top10pct': 0.612, 'precision_top10pct': 0.41, 'business_value_usd_estimado': 240_000},
        stack=['python', 'lightgbm', 'pandas', 'shap', 'mlflow', 'evidently'],
        repo_url='https://github.com/usuario/capstone-churn',
        demo_url=None,
        blog_url='https://usuario.github.io/blog/2026/churn-saas/',
        limitations=['Train test split temporal: drift visible más allá de 6 meses.', 'No incluye señales de soporte (tickets).', 'Coste de FN >> FP no se modela explícitamente todavía.'],
        hero_image='assets/churn-hero.png',
    ),
]
print(f'Proyectos cargados: {len(projects)}')
for p in projects: print(f'  - {p.slug}: {len(p.metrics)} métricas, {len(p.limitations)} limitaciones')

## 2. `render_project_md(project)` — página de proyecto

In [ ]:
def render_project_md(p: Project) -> str:
    badges = f'![CI](https://github.com/usuario/{p.slug}/actions/workflows/ci.yml/badge.svg) ![License](https://img.shields.io/badge/license-MIT-blue) ![Python](https://img.shields.io/badge/python-3.12-blue)'
    hero = f'![hero]({p.hero_image})\n\n' if p.hero_image else ''
    metrics_table = '| Métrica | Valor |\n|---|---|\n' + '\n'.join(f'| `{k}` | **{v}** |' for k, v in p.metrics.items())
    demo_line = f'\n**Demo en vivo:** [{p.demo_url}]({p.demo_url})\n' if p.demo_url else '\n_Sin demo hosted; ver video en el repo._\n'
    blog_line = f'\n**Post técnico:** [{p.blog_url}]({p.blog_url})\n' if p.blog_url else ''
    lims = '\n'.join(f'- {l}' for l in p.limitations)
    stack_line = ', '.join(f'`{s}`' for s in p.stack)
    return f"""# {p.title}

{badges}

{hero}## Problema

{p.problem}

## Approach

{p.approach}

## Stack

{stack_line}

## Resultados

{metrics_table}
{demo_line}{blog_line}
## Quick start (30 seg)

```bash
git clone {p.repo_url}
cd {p.slug}
make demo   # levanta el modelo local en localhost:8000
```

## Limitaciones conocidas

{lims}

## Citation

Ver `CITATION.cff` en el repo.
"""

sample_md = render_project_md(projects[0])
print(sample_md[:900])
print('...')

## 3. `render_index_md(projects, author)` — landing del portafolio

In [ ]:
def render_index_md(projects, author: dict) -> str:
    cards = []
    for p in projects:
        main_metric = next(iter(p.metrics.items()))
        cards.append(f'### [{p.title}](projects/{p.slug}.md)\n\n{p.problem}\n\n**{main_metric[0]} = {main_metric[1]}** · [repo]({p.repo_url})' + (f' · [demo]({p.demo_url})' if p.demo_url else ''))
    cards_md = '\n\n---\n\n'.join(cards)
    return f"""# {author['name']}

{author['tagline']}

{author['bio']}

[GitHub]({author['github']}) · [LinkedIn]({author['linkedin']}) · [Blog](blog/) · [CV (PDF)](cv.pdf)

## Proyectos

{cards_md}
"""

author = {
    'name': 'Nombre Apellido',
    'tagline': '_Data Scientist · NLP · MLOps_',
    'bio': 'Construyo modelos que llegan a producción. Ex-ingeniero industrial, ahora trabajo con NLP en español y series temporales.',
    'github': 'https://github.com/usuario',
    'linkedin': 'https://linkedin.com/in/usuario',
}
print(render_index_md(projects, author))

## 4. `validate_portfolio(projects)` — gate de calidad

In [ ]:
PLACEHOLDER_TOKENS = ['lorem', 'tbd', 'todo', 'xxx', 'placeholder', 'descripción del proyecto']

def validate_portfolio(projects) -> list:
    """Devuelve lista de errores. Lista vacía == portafolio listo para publicar."""
    errors = []
    if len(projects) < 3:
        errors.append(f'FAIL min_projects: {len(projects)} < 3.')
    for p in projects:
        # (b) métrica numérica
        numeric = [v for v in p.metrics.values() if isinstance(v, (int, float))]
        if not numeric:
            errors.append(f'[{p.slug}] sin métrica numérica.')
        # (c) repo_url
        if not p.repo_url or not p.repo_url.startswith('http'):
            errors.append(f'[{p.slug}] repo_url inválido.')
        # (d) descripciones no placeholder
        full_text = (p.problem + ' ' + p.approach).lower()
        if any(tok in full_text for tok in PLACEHOLDER_TOKENS):
            errors.append(f'[{p.slug}] contiene texto placeholder.')
        if len(p.problem) < 40:
            errors.append(f'[{p.slug}] problem demasiado corto ({len(p.problem)} chars).')
        if not p.limitations:
            errors.append(f'[{p.slug}] sin limitaciones declaradas (red flag de honestidad).')
    return errors

errors = validate_portfolio(projects)
print('Errores:', errors if errors else 'NINGUNO — portafolio listo.')

## 5. Model Card mínima (Mitchell et al. 2019)

In [ ]:
def render_model_card(p: Project) -> str:
    metrics_lines = '\n'.join(f'- **{k}**: {v}' for k, v in p.metrics.items())
    return f"""# Model Card — {p.title}

## Detalles del modelo
- Repo: {p.repo_url}
- Stack: {', '.join(p.stack)}

## Uso previsto
{p.problem}

## Uso NO previsto
- Decisiones de alto impacto (clínicas, judiciales, financieras vinculantes) sin human-in-the-loop.
- Generalización fuera del dominio descripto.

## Datos de entrenamiento
Ver `data/README.md` en el repo (fuentes, licencias, fecha de captura).

## Métricas
{metrics_lines}

## Limitaciones y consideraciones éticas
{chr(10).join('- ' + l for l in p.limitations)}

## Citation
Mitchell et al., *Model Cards for Model Reporting*, FAT* 2019.
"""

print(render_model_card(projects[1]))

## 6. Estructura del sitio: `mkdocs.yml` + tree

In [ ]:
MKDOCS_YML = textwrap.dedent('''
site_name: Nombre Apellido — Portfolio
site_url: https://usuario.github.io/portfolio/
repo_url: https://github.com/usuario/portfolio

theme:
  name: material
  features:
    - navigation.tabs
    - navigation.sections
    - search.suggest
    - content.code.copy
  palette:
    - scheme: default
      primary: indigo
      toggle: {icon: material/weather-sunny, name: dark mode}
    - scheme: slate
      primary: indigo
      toggle: {icon: material/weather-night, name: light mode}

nav:
  - Inicio: index.md
  - Proyectos:
      - NLP sentimiento ES: projects/nlp-sentimiento-es.md
      - CV defect detection: projects/cv-defect-detection.md
      - Churn SaaS: projects/tabular-churn.md
  - Blog: blog/index.md
  - CV: cv.md

plugins:
  - search
  - blog

markdown_extensions:
  - admonition
  - pymdownx.highlight
  - pymdownx.superfences
''').strip()

TREE = textwrap.dedent('''
portfolio/
├── mkdocs.yml
├── docs/
│   ├── index.md
│   ├── cv.md
│   ├── cv.pdf
│   ├── assets/
│   │   ├── nlp-hero.gif
│   │   ├── cv-hero.png
│   │   └── churn-hero.png
│   ├── projects/
│   │   ├── nlp-sentimiento-es.md
│   │   ├── cv-defect-detection.md
│   │   └── tabular-churn.md
│   └── blog/
│       ├── index.md
│       └── posts/
│           ├── 2026-sentimiento-es.md
│           ├── 2026-cv-defects.md
│           └── 2026-churn-saas.md
└── .github/workflows/pages.yml
''').strip()

print(TREE)
print('\n--- mkdocs.yml ---')
print(MKDOCS_YML)

## 7. GitHub Actions workflow para deploy automático

In [ ]:
PAGES_WORKFLOW = textwrap.dedent('''
name: deploy-portfolio

on:
  push:
    branches: [main]
  workflow_dispatch:

permissions:
  contents: read
  pages: write
  id-token: write

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with: {python-version: "3.12"}
      - run: pip install mkdocs-material
      - run: mkdocs build --strict
      - uses: actions/upload-pages-artifact@v3
        with: {path: site}

  deploy:
    needs: build
    runs-on: ubuntu-latest
    environment:
      name: github-pages
      url: ${{ steps.deployment.outputs.page_url }}
    steps:
      - id: deployment
        uses: actions/deploy-pages@v4
''').strip()
print(PAGES_WORKFLOW)

## 8. Outline del deck de presentación (10-15 slides)

In [ ]:
DECK_OUTLINE = [
    ('01', 'Portada', 'Nombre, rol target, foto, URL del portafolio.'),
    ('02', 'Quién soy en 30 seg', 'Background, vertical, qué busco.'),
    ('03', 'Proyecto 1 — Problema', 'Una frase humana + por qué importa.'),
    ('04', 'Proyecto 1 — Datos + approach', 'Diagrama del pipeline (1 imagen).'),
    ('05', 'Proyecto 1 — Resultado', 'Métrica vs baseline + 1 plot.'),
    ('06', 'Proyecto 2 — Problema', 'Mismo formato.'),
    ('07', 'Proyecto 2 — Approach + resultado', 'Combinar para densidad.'),
    ('08', 'Proyecto 3 — Problema', 'Mismo formato.'),
    ('09', 'Proyecto 3 — Approach + resultado', 'Combinar.'),
    ('10', 'Stack común', 'Logos: Python, PyTorch, LightGBM, Docker, GH Actions.'),
    ('11', 'Cómo trabajo', '3 bullets: tests, lock files, CI verde, docs.'),
    ('12', 'Trade-offs aprendidos', 'Lo que NO funcionó. Es lo más valioso del deck.'),
    ('13', 'Próximos pasos', 'En qué estoy trabajando ahora.'),
    ('14', 'Contacto', 'Mail, LinkedIn, GitHub, URL portafolio. QR opcional.'),
]
for n, t, d in DECK_OUTLINE: print(f'  {n}. {t:<32} — {d}')

## 9. Outline del blog post técnico

In [ ]:
BLOG_OUTLINE = textwrap.dedent('''
1. **Hook (1 párrafo)**: anécdota o número que sorprende. Ej: "De 1000 reseñas analizadas, 38% que parecían positivas eran sarcasmo."
2. **Problema** (1-2 párrafos): qué dolor resuelve. Lenguaje de negocio, no técnico.
3. **Datos** (1-2 párrafos + 1 tabla): fuente, tamaño, label process, distribución.
4. **Approach** (2-4 párrafos): por qué este modelo y no otro. Mencionar 1 alternativa que NO usaste y por qué.
5. **Resultado clave + plot** (1 párrafo + 1 viz): UNA métrica primaria + 1 gráfico (no más).
6. **Trade-offs** (2-3 bullets): qué sacrificaste. Honestidad = credibilidad.
7. **Próximos pasos** (3 bullets): qué harías con más tiempo/datos.
8. **Call to action**: link al repo, link a la demo, invitar a abrir un issue.
''').strip()
print(BLOG_OUTLINE)

## 10. Checklist final pre-publicación (15 ítems)

In [ ]:
CHECKLIST = [
    'index.md con nombre, tagline, foto, bio de 3 líneas.',
    '3 proyectos linkeados desde el index.',
    'Cada proyecto tiene README con badges (CI, license, python).',
    'Cada proyecto tiene hero image o GIF.',
    'Cada proyecto tiene métrica numérica primaria visible above-the-fold.',
    'Cada proyecto tiene quick start de 30 seg que realmente funciona.',
    'Cada proyecto tiene sección "Limitaciones" con >= 3 bullets honestos.',
    'Al menos 1 proyecto con demo hosted funcionando hoy (probada en navegador incógnito).',
    'Blog con al menos 1 post técnico publicado.',
    'CV PDF de 1 página linkeado desde el index.',
    'Lock files (uv.lock / poetry.lock) committeados en cada repo.',
    'CI verde en cada repo (badge clickeable).',
    'Sitio responsive: probado en mobile.',
    'Links externos abren sin error (verificar con `lychee` o linkchecker).',
    'Portafolio review por 1 persona NO técnica: ¿entiende qué hacés en 30 seg?',
]
for i, item in enumerate(CHECKLIST, 1): print(f'  [{i:2}] {item}')
print(f'\n{len(CHECKLIST)} ítems verificables.')

## 11. Exportar todo a disco (dry run)

In [ ]:
# Simulamos el árbol de archivos que se generaría. No escribimos a disco —
# solo mostramos el mapa { path: tamaño_en_chars } para verificar coherencia.

outputs = {'mkdocs.yml': MKDOCS_YML, 'docs/index.md': render_index_md(projects, author), '.github/workflows/pages.yml': PAGES_WORKFLOW}
for p in projects:
    outputs[f'docs/projects/{p.slug}.md'] = render_project_md(p)
    outputs[f'docs/projects/{p.slug}-model-card.md'] = render_model_card(p)

for path, content in sorted(outputs.items()):
    print(f'  {path:<55} {len(content):>5} chars')
print(f'\nTOTAL: {len(outputs)} archivos, {sum(len(c) for c in outputs.values()):,} chars.')
errors = validate_portfolio(projects)
print(f'\nValidación final: {"OK — publicar" if not errors else f"FAIL — {len(errors)} errores"}')

## 🎓 Cierre del programa

Esta es la celda final del notebook final de la clase final. Llegaste.

**Especializaciones recomendadas** (elegí UNA y profundizá 6-12 meses):

- **Research / Applied Science**: papers, NeurIPS/ICML reading group, contribuir a una librería open source (HF, scikit-learn). Doctorado opcional.
- **Applied ML / MLE**: sistemas de inferencia a escala, feature stores, serving con Ray/BentoML, observabilidad de modelos en prod.
- **Product DS**: experimentación (A/B test, multi-armed bandits, causal inference), métricas de producto, comunicación con stakeholders.
- **Data Engineering / Platform**: Spark/Flink, dbt, Airflow/Dagster, lakehouse, gobierno de datos.

**Hábitos que sostienen una carrera técnica:**

1. **Enviá** algo público cada trimestre (proyecto, post, charla, PR a OSS).
2. **Leé** 1 paper o post técnico largo por semana.
3. **Mostrá tu trabajo** antes de creer que está listo — el feedback temprano vale más que el pulido tardío.
4. **Documentá decisiones**, no solo código. Tu yo de dentro de 6 meses te lo agradece.
5. **Enseñá** lo que recién aprendiste. Es la forma más rápida de consolidarlo.

Fin del programa. Ahora andá a romperla. 🎓